# 🛒 Bodega de Datos — Customer Shopping Dataset
## Proyecto Académico | Ingeniería de Datos

| | |
|---|---|
| **Dataset** | Customer Shopping Dataset — Kaggle |
| **Modelo** | Estrella (Star Schema) |
| **Base de datos** | PostgreSQL |
| **Herramientas** | Python · Pandas · SQLAlchemy · Matplotlib · Seaborn |

---

## Estructura del proyecto

| Fase | Descripción | Peso |
|------|-------------|------|
| **Fase 1** | Diseño del modelo de bodega de datos | 20% |
| **Fase 2** | ETL (Extracción, Transformación y Carga) | 30% |
| **Fase 3** | Consultas analíticas SQL | 20% |
| **Fase 4** | Análisis descriptivo y visualización | 20% |


---
# Fase 1 — Diseño del Modelo de Bodega de Datos

## 1.1 Análisis del dataset

El **Customer Shopping Dataset** contiene **99.457 registros** de transacciones de compra realizadas en centros comerciales de Turquía, con las siguientes variables:

| Columna | Tipo | Descripción |
|---------|------|-------------|
| `invoice_no` | Texto | Identificador único de la factura |
| `customer_id` | Texto | Identificador único del cliente |
| `gender` | Categórico | Género del cliente |
| `age` | Numérico | Edad del cliente |
| `category` | Categórico | Categoría del producto |
| `quantity` | Numérico | Cantidad comprada |
| `price` | Numérico | Precio unitario |
| `payment_method` | Categórico | Método de pago |
| `invoice_date` | Fecha | Fecha de la transacción |
| `shopping_mall` | Categórico | Centro comercial |

## 1.2 Decisión del modelo: ⭐ Estrella (Star Schema)

Se eligió el **modelo estrella** por las siguientes razones:

1. **Simplicidad de consultas:** Las dimensiones se unen directamente a la tabla de hechos sin joins adicionales entre dimensiones, lo que reduce la complejidad de las consultas analíticas.
2. **Rendimiento:** Menor número de joins significa mayor velocidad en consultas OLAP sobre ~100k registros.
3. **Dimensiones simples:** Ninguna dimensión requiere jerarquías complejas que justifiquen un copo de nieve. Por ejemplo, `shopping_mall` no tiene atributos adicionales como ciudad o región que necesiten normalizarse.
4. **Facilidad de mantenimiento:** El modelo estrella es más fácil de entender y mantener para equipos de análisis de negocio.

> El **modelo copo de nieve** se descartó porque añadiría complejidad (más joins) sin beneficio real dado el tamaño y estructura del dataset.

## 1.3 Diagrama del modelo estrella

```
                    ┌─────────────────┐
                    │   dim_customer  │
                    │─────────────────│
                    │ customer_key PK │
                    │ customer_id     │
                    │ gender          │
                    │ age             │
                    └────────┬────────┘
                             │
┌──────────────┐    ┌────────▼─────────────────────┐    ┌─────────────────┐
│  dim_date    │    │         fact_sales            │    │  dim_category   │
│──────────────│    │──────────────────────────────-│    │─────────────────│
│ date_key PK  │◄───│ sale_id      PK               │    │ category_key PK │
│ full_date    │    │ invoice_no                    │───►│ category_name   │
│ day          │    │ customer_key FK               │    └─────────────────┘
│ month        │    │ date_key     FK               │
│ year         │    │ category_key FK               │    ┌─────────────────┐
│ is_weekend   │    │ payment_key  FK               │    │  dim_payment    │
└──────────────┘    │ mall_key     FK               │───►│─────────────────│
                    │ quantity                      │    │ payment_key PK  │
                    │ price                         │    │ payment_method  │
                    │ total_amount                  │    └─────────────────┘
                    └──────────────┬────────────────┘
                                   │            ┌─────────────────┐
                                   └───────────►│   dim_mall      │
                                                │─────────────────│
                                                │ mall_key PK     │
                                                │ shopping_mall   │
                                                └─────────────────┘
```

## 1.4 DDL — Implementación en PostgreSQL

El siguiente bloque contiene el script SQL para crear el esquema completo en PostgreSQL.


In [ ]:
# DDL del modelo estrella — ejecutar directamente en PostgreSQL
# (Se muestra aquí para documentación; la creación de tablas se gestiona desde la BD)

ddl_script = """
-- ============================================================
-- MODELO ESTRELLA: Customer Shopping Data Warehouse
-- ============================================================

-- Dimensión Cliente
CREATE TABLE IF NOT EXISTS dim_customer (
    customer_key  SERIAL PRIMARY KEY,
    customer_id   VARCHAR(20) NOT NULL UNIQUE,
    gender        VARCHAR(10),
    age           INTEGER
);

-- Dimensión Categoría de Producto
CREATE TABLE IF NOT EXISTS dim_category (
    category_key  SERIAL PRIMARY KEY,
    category_name VARCHAR(50) NOT NULL UNIQUE
);

-- Dimensión Método de Pago
CREATE TABLE IF NOT EXISTS dim_payment (
    payment_key    SERIAL PRIMARY KEY,
    payment_method VARCHAR(30) NOT NULL UNIQUE
);

-- Dimensión Centro Comercial
CREATE TABLE IF NOT EXISTS dim_mall (
    mall_key      SERIAL PRIMARY KEY,
    shopping_mall VARCHAR(100) NOT NULL UNIQUE
);

-- Dimensión Fecha
CREATE TABLE IF NOT EXISTS dim_date (
    date_key    SERIAL PRIMARY KEY,
    full_date   DATE NOT NULL UNIQUE,
    day         INTEGER,
    month       INTEGER,
    year        INTEGER,
    is_weekend  BOOLEAN
);

-- Tabla de Hechos
CREATE TABLE IF NOT EXISTS fact_sales (
    sale_id       SERIAL PRIMARY KEY,
    invoice_no    VARCHAR(20),
    customer_key  INTEGER REFERENCES dim_customer(customer_key),
    date_key      INTEGER REFERENCES dim_date(date_key),
    category_key  INTEGER REFERENCES dim_category(category_key),
    payment_key   INTEGER REFERENCES dim_payment(payment_key),
    mall_key      INTEGER REFERENCES dim_mall(mall_key),
    quantity      INTEGER,
    price         NUMERIC(12,2),
    total_amount  NUMERIC(12,2)
);
"""

print("Script DDL del modelo estrella:")
print(ddl_script)


---
# Fase 2 — ETL (Extracción, Transformación y Carga)

El proceso ETL se divide en tres etapas principales:

1. **Extracción:** Lectura del dataset CSV con Pandas
2. **Transformación:** Limpieza, conversión de tipos y creación de dimensiones
3. **Carga:** Inserción en PostgreSQL mediante SQLAlchemy

## 2.1 Importación de librerías


In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import os

warnings.filterwarnings("ignore")

# Configuración global de visualizaciones
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["font.size"] = 11
sns.set_theme(style="whitegrid", palette="muted")

print("✅ Librerías importadas correctamente")


## 2.2 Configuración de la conexión a PostgreSQL

> ⚠️ **Buena práctica de seguridad:** Las credenciales se leen desde variables de entorno en lugar de estar escritas en el código.  
> Antes de ejecutar, define las variables en tu terminal:
> ```bash
> export DB_USER=postgres
> export DB_PASSWORD=tu_contraseña
> export DB_NAME=modelo_estrella_ventas
> ```


In [ ]:
# ─── Configuración de conexión (credenciales desde variables de entorno) ───
DB_USER     = os.getenv("DB_USER",     "postgres")
DB_PASSWORD = os.getenv("DB_PASSWORD", "tu_contraseña_aqui")   # ← cambia solo localmente
DB_HOST     = os.getenv("DB_HOST",     "localhost")
DB_PORT     = os.getenv("DB_PORT",     "5432")
DB_NAME     = os.getenv("DB_NAME",     "modelo_estrella_ventas")

engine = create_engine(
    f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}",
    pool_pre_ping=True   # verifica la conexión antes de usarla
)

# Verificar conexión
try:
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
    print("✅ Conexión a PostgreSQL exitosa")
except Exception as e:
    print(f"❌ Error de conexión: {e}")


## 2.3 Extracción — Carga del dataset

In [ ]:
# ─── EXTRACCIÓN ───────────────────────────────────────────────────────────────
df = pd.read_csv("customer_shopping_data.csv")

print(f"✅ Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")
df.head(5)


## 2.4 Análisis exploratorio inicial (EDA)

In [ ]:
# ─── ESTADÍSTICAS DESCRIPTIVAS ────────────────────────────────────────────────
print("=" * 55)
print("RESUMEN ESTADÍSTICO — VARIABLES NUMÉRICAS")
print("=" * 55)
display(df.describe())

print("\n" + "=" * 55)
print("ESTRUCTURA DEL DATASET")
print("=" * 55)
df.info()

print("\n" + "=" * 55)
print("VALORES NULOS POR COLUMNA")
print("=" * 55)
print(df.isnull().sum())


## 2.5 Transformación — Limpieza y preparación de datos

In [ ]:
# ─── TRANSFORMACIÓN ───────────────────────────────────────────────────────────

# 1. Variables categóricas
categorical_cols = ["gender", "payment_method", "category", "shopping_mall"]
df[categorical_cols] = df[categorical_cols].astype("category")

# 2. Conversión de fecha (formato DD/MM/YYYY)
df["invoice_date"] = pd.to_datetime(df["invoice_date"], dayfirst=True)

# 3. Componentes de fecha
df["day"]        = df["invoice_date"].dt.day.astype("int32")
df["month"]      = df["invoice_date"].dt.month.astype("int32")
df["year"]       = df["invoice_date"].dt.year.astype("int32")
df["is_weekend"] = df["invoice_date"].dt.weekday >= 5

# 4. Métrica principal de la tabla de hechos
df["total_amount"] = df["quantity"] * df["price"]

# 5. Valores únicos por variable categórica
print("Valores únicos por variable categórica:")
for col in categorical_cols:
    print(f"  {col}: {df[col].unique().tolist()}")

# 6. Verificación de duplicados
dupes = df[df.duplicated(subset=["customer_id", "invoice_date", "total_amount"])]
print(f"\n{'✅' if len(dupes) == 0 else '⚠️'} Registros duplicados encontrados: {len(dupes)}")

print("\n✅ Transformaciones completadas")
df.info()


## 2.6 Carga — Construcción y carga de tablas de dimensiones

Cada dimensión se construye extrayendo los valores únicos del DataFrame y añadiendo una clave surrogada (`_key`) como identificador único en la bodega.


In [ ]:
# ─── FUNCIÓN DE CARGA CON MANEJO DE ERRORES ──────────────────────────────────

def cargar_tabla(df_dim, nombre_tabla, engine):
    """
    Carga un DataFrame en PostgreSQL usando modo 'append'.

    Args:
        df_dim (pd.DataFrame): DataFrame con los datos a cargar.
        nombre_tabla (str): Nombre de la tabla destino en PostgreSQL.
        engine: Motor de conexión SQLAlchemy.

    Returns:
        int: Número de filas cargadas, o -1 si ocurrió un error.
    """
    try:
        df_dim.to_sql(nombre_tabla, engine, if_exists="append", index=False)
        print(f"  ✅ {nombre_tabla}: {len(df_dim):,} filas cargadas")
        return len(df_dim)
    except Exception as e:
        print(f"  ❌ Error cargando {nombre_tabla}: {e}")
        return -1


# ─── CONSTRUCCIÓN DE DIMENSIONES ──────────────────────────────────────────────

dim_customer = (
    df[["customer_id", "gender", "age"]]
    .drop_duplicates(subset=["customer_id"])
    .reset_index(drop=True)
)

dim_category = (
    df[["category"]]
    .drop_duplicates()
    .reset_index(drop=True)
    .rename(columns={"category": "category_name"})
)

dim_payment = (
    df[["payment_method"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_mall = (
    df[["shopping_mall"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_date = (
    df[["invoice_date", "day", "month", "year", "is_weekend"]]
    .drop_duplicates(subset=["invoice_date"])
    .reset_index(drop=True)
    .rename(columns={"invoice_date": "full_date"})
)

print("Dimensiones construidas:")
print(f"  dim_customer : {len(dim_customer):,} registros únicos")
print(f"  dim_category : {len(dim_category):,} categorías")
print(f"  dim_payment  : {len(dim_payment):,} métodos de pago")
print(f"  dim_mall     : {len(dim_mall):,} centros comerciales")
print(f"  dim_date     : {len(dim_date):,} fechas únicas")


In [ ]:
# ─── CARGA DE DIMENSIONES EN POSTGRESQL ──────────────────────────────────────

print("Cargando dimensiones en PostgreSQL...")
cargar_tabla(dim_customer, "dim_customer", engine)
cargar_tabla(dim_category, "dim_category", engine)
cargar_tabla(dim_payment,  "dim_payment",  engine)
cargar_tabla(dim_mall,     "dim_mall",     engine)
cargar_tabla(dim_date,     "dim_date",     engine)
print("\n✅ Todas las dimensiones cargadas")


## 2.7 Carga — Construcción y carga de la tabla de hechos

Se recuperan las claves surrogadas generadas por PostgreSQL y se usan para construir `fact_sales`.


In [ ]:
# ─── RECUPERAR DIMENSIONES CON CLAVES SURROGADAS ─────────────────────────────

dim_customer_db = pd.read_sql("SELECT * FROM dim_customer", engine)
dim_category_db = pd.read_sql("SELECT * FROM dim_category", engine)
dim_payment_db  = pd.read_sql("SELECT * FROM dim_payment",  engine)
dim_mall_db     = pd.read_sql("SELECT * FROM dim_mall",     engine)
dim_date_db     = pd.read_sql("SELECT * FROM dim_date",     engine)

dim_date_db["full_date"] = pd.to_datetime(dim_date_db["full_date"])

print("✅ Dimensiones recuperadas desde PostgreSQL")
print(f"  Columnas dim_customer : {dim_customer_db.columns.tolist()}")
print(f"  Columnas dim_category : {dim_category_db.columns.tolist()}")
print(f"  Columnas dim_payment  : {dim_payment_db.columns.tolist()}")
print(f"  Columnas dim_mall     : {dim_mall_db.columns.tolist()}")
print(f"  Columnas dim_date     : {dim_date_db.columns.tolist()}")


In [ ]:
# ─── CONSTRUCCIÓN DE FACT_SALES ──────────────────────────────────────────────

fact_sales = (
    df
    .merge(dim_customer_db[["customer_key","customer_id"]], on="customer_id", how="left")
    .merge(dim_category_db[["category_key","category_name"]],
           left_on="category", right_on="category_name", how="left")
    .merge(dim_payment_db[["payment_key","payment_method"]], on="payment_method", how="left")
    .merge(dim_mall_db[["mall_key","shopping_mall"]], on="shopping_mall", how="left")
    .merge(dim_date_db[["date_key","full_date"]],
           left_on="invoice_date", right_on="full_date", how="left")
)

fact_sales = fact_sales[[
    "invoice_no", "customer_key", "date_key",
    "category_key", "payment_key", "mall_key",
    "quantity", "price", "total_amount"
]]

# Verificar que no hayan claves nulas (indicaría un problema de join)
nulos = fact_sales[["customer_key","date_key","category_key","payment_key","mall_key"]].isnull().sum()
print("Claves nulas en fact_sales:")
print(nulos)

print(f"\n✅ fact_sales construida: {len(fact_sales):,} filas")
fact_sales.head(3)


In [ ]:
# ─── CARGA DE FACT_SALES ─────────────────────────────────────────────────────

cargar_tabla(fact_sales, "fact_sales", engine)
print("\n🎉 ETL COMPLETADO — Datos cargados correctamente en PostgreSQL")


## 2.8 Verificación de datos insertados

In [ ]:
# ─── VERIFICACIÓN POST-CARGA ──────────────────────────────────────────────────

tablas = ["dim_customer", "dim_category", "dim_payment", "dim_mall", "dim_date", "fact_sales"]

print("=" * 45)
print("VERIFICACIÓN DE REGISTROS EN POSTGRESQL")
print("=" * 45)
for tabla in tablas:
    try:
        n = pd.read_sql(f"SELECT COUNT(*) as n FROM {tabla}", engine)["n"][0]
        print(f"  {tabla:<20} : {n:>8,} registros")
    except Exception as e:
        print(f"  {tabla:<20} : ❌ {e}")


---
# Fase 3 — Consultas Analíticas SQL

Se presentan las 5 consultas analíticas requeridas, ejecutadas directamente contra el Data Warehouse en PostgreSQL.

## Estrategia de optimización con índices

Antes de ejecutar las consultas, se crean índices en las columnas de join y filtro más frecuentes para mejorar el rendimiento.


In [ ]:
# ─── CREACIÓN DE ÍNDICES PARA OPTIMIZACIÓN ───────────────────────────────────

indices = [
    "CREATE INDEX IF NOT EXISTS idx_fact_customer  ON fact_sales(customer_key);",
    "CREATE INDEX IF NOT EXISTS idx_fact_date       ON fact_sales(date_key);",
    "CREATE INDEX IF NOT EXISTS idx_fact_category   ON fact_sales(category_key);",
    "CREATE INDEX IF NOT EXISTS idx_fact_payment    ON fact_sales(payment_key);",
    "CREATE INDEX IF NOT EXISTS idx_fact_mall       ON fact_sales(mall_key);",
    "CREATE INDEX IF NOT EXISTS idx_date_month_year ON dim_date(month, year);",
]

with engine.connect() as conn:
    for idx in indices:
        try:
            conn.execute(text(idx))
            conn.commit()
            print(f"  ✅ {idx.split('idx_')[1].split(' ')[0]}")
        except Exception as e:
            print(f"  ⚠️ {e}")

print("\n✅ Índices creados")


## Consulta 1 — Total de ventas por categoría de producto

In [ ]:
# ─── CONSULTA 1: VENTAS POR CATEGORÍA ────────────────────────────────────────

query_1 = """
    SELECT
        c.category_name,
        COUNT(f.invoice_no)          AS num_transacciones,
        SUM(f.quantity)              AS unidades_vendidas,
        ROUND(SUM(f.total_amount)::numeric, 2) AS total_ventas,
        ROUND(AVG(f.total_amount)::numeric, 2) AS ticket_promedio
    FROM fact_sales f
    JOIN dim_category c ON f.category_key = c.category_key
    GROUP BY c.category_name
    ORDER BY total_ventas DESC;
"""

df_q1 = pd.read_sql(query_1, engine)
print("Total de ventas por categoría:")
display(df_q1)


## Consulta 2 — Clientes con mayor volumen de compras

In [ ]:
# ─── CONSULTA 2: TOP 10 CLIENTES ─────────────────────────────────────────────

query_2 = """
    SELECT
        cu.customer_id,
        cu.gender,
        cu.age,
        COUNT(f.invoice_no)          AS num_compras,
        SUM(f.quantity)              AS unidades_totales,
        ROUND(SUM(f.total_amount)::numeric, 2) AS gasto_total
    FROM fact_sales f
    JOIN dim_customer cu ON f.customer_key = cu.customer_key
    GROUP BY cu.customer_id, cu.gender, cu.age
    ORDER BY gasto_total DESC
    LIMIT 10;
"""

df_q2 = pd.read_sql(query_2, engine)
print("Top 10 clientes por volumen de gasto:")
display(df_q2)


## Consulta 3 — Métodos de pago más utilizados

In [ ]:
# ─── CONSULTA 3: MÉTODOS DE PAGO ─────────────────────────────────────────────

query_3 = """
    SELECT
        p.payment_method,
        COUNT(f.invoice_no)          AS num_transacciones,
        ROUND(SUM(f.total_amount)::numeric, 2)  AS total_ventas,
        ROUND(
            100.0 * COUNT(f.invoice_no) / SUM(COUNT(f.invoice_no)) OVER (),
            2
        )                            AS porcentaje_uso
    FROM fact_sales f
    JOIN dim_payment p ON f.payment_key = p.payment_key
    GROUP BY p.payment_method
    ORDER BY num_transacciones DESC;
"""

df_q3 = pd.read_sql(query_3, engine)
print("Métodos de pago más utilizados:")
display(df_q3)


## Consulta 4 — Comparación de ventas por mes

In [ ]:
# ─── CONSULTA 4: VENTAS POR MES ──────────────────────────────────────────────

query_4 = """
    SELECT
        d.year,
        d.month,
        TO_CHAR(TO_DATE(d.month::text, 'MM'), 'Month') AS nombre_mes,
        COUNT(f.invoice_no)          AS num_transacciones,
        ROUND(SUM(f.total_amount)::numeric, 2) AS total_ventas
    FROM fact_sales f
    JOIN dim_date d ON f.date_key = d.date_key
    GROUP BY d.year, d.month
    ORDER BY d.year, d.month;
"""

df_q4 = pd.read_sql(query_4, engine)
print("Ventas mensuales:")
display(df_q4)


## Consulta 5 — Ventas por centro comercial y comparación semana vs fin de semana

In [ ]:
# ─── CONSULTA 5: VENTAS POR MALL Y DÍA ──────────────────────────────────────

query_5 = """
    SELECT
        m.shopping_mall,
        CASE WHEN d.is_weekend THEN 'Fin de semana' ELSE 'Entre semana' END AS tipo_dia,
        COUNT(f.invoice_no)          AS num_transacciones,
        ROUND(SUM(f.total_amount)::numeric, 2) AS total_ventas
    FROM fact_sales f
    JOIN dim_mall m ON f.mall_key = m.mall_key
    JOIN dim_date d ON f.date_key = d.date_key
    GROUP BY m.shopping_mall, d.is_weekend
    ORDER BY m.shopping_mall, d.is_weekend;
"""

df_q5 = pd.read_sql(query_5, engine)
print("Ventas por centro comercial y tipo de día:")
display(df_q5)


---
# Fase 4 — Análisis Descriptivo y Visualización

## 4.1 Distribución de ventas por categoría


In [ ]:
# ─── GRÁFICA 1: VENTAS POR CATEGORÍA (BARRAS HORIZONTALES) ──────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Ventas totales
df_q1_sorted = df_q1.sort_values("total_ventas", ascending=True)
bars = axes[0].barh(df_q1_sorted["category_name"], df_q1_sorted["total_ventas"],
                    color=sns.color_palette("muted", len(df_q1_sorted)))
axes[0].set_xlabel("Total de Ventas (USD)")
axes[0].set_title("Total de Ventas por Categoría")
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
for bar in bars:
    axes[0].text(bar.get_width() + bar.get_width()*0.01, bar.get_y() + bar.get_height()/2,
                 f"${bar.get_width():,.0f}", va="center", fontsize=9)

# Ticket promedio
df_q1_sorted2 = df_q1.sort_values("ticket_promedio", ascending=True)
axes[1].barh(df_q1_sorted2["category_name"], df_q1_sorted2["ticket_promedio"],
             color=sns.color_palette("pastel", len(df_q1_sorted2)))
axes[1].set_xlabel("Ticket Promedio (USD)")
axes[1].set_title("Ticket Promedio por Categoría")
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))

plt.suptitle("Análisis de Ventas por Categoría de Producto", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("fig1_ventas_categoria.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Figura 1 guardada")


**📊 Insight:** Las categorías con mayor volumen de ventas revelan los productos más demandados en los centros comerciales. La comparación entre ventas totales y ticket promedio permite identificar si una categoría vende mucho por volumen o por precio unitario.


## 4.2 Evolución temporal de ventas

In [ ]:
# ─── GRÁFICA 2: TENDENCIA MENSUAL DE VENTAS ──────────────────────────────────

df_q4["periodo"] = df_q4["year"].astype(str) + "-" + df_q4["month"].astype(str).str.zfill(2)

fig, axes = plt.subplots(2, 1, figsize=(14, 9))

# Ventas totales por mes
axes[0].plot(df_q4["periodo"], df_q4["total_ventas"],
             marker="o", linewidth=2, color="#2196F3", markersize=5)
axes[0].fill_between(range(len(df_q4)), df_q4["total_ventas"], alpha=0.15, color="#2196F3")
axes[0].set_xticks(range(len(df_q4)))
axes[0].set_xticklabels(df_q4["periodo"], rotation=45, ha="right", fontsize=8)
axes[0].set_title("Evolución Mensual de Ventas Totales", fontweight="bold")
axes[0].set_ylabel("Ventas Totales (USD)")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))

# Número de transacciones por mes
axes[1].bar(range(len(df_q4)), df_q4["num_transacciones"],
            color=sns.color_palette("Blues_d", len(df_q4)), width=0.7)
axes[1].set_xticks(range(len(df_q4)))
axes[1].set_xticklabels(df_q4["periodo"], rotation=45, ha="right", fontsize=8)
axes[1].set_title("Número de Transacciones por Mes", fontweight="bold")
axes[1].set_ylabel("Transacciones")

plt.suptitle("Tendencia Temporal de Ventas", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("fig2_tendencia_mensual.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Figura 2 guardada")


**📊 Insight:** La tendencia mensual permite identificar estacionalidad en las compras, picos de venta y meses de baja actividad, información clave para la planificación de inventario y campañas de marketing.


## 4.3 Métodos de pago

In [ ]:
# ─── GRÁFICA 3: MÉTODOS DE PAGO ──────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = sns.color_palette("Set2", len(df_q3))

# Pie chart — participación
axes[0].pie(
    df_q3["num_transacciones"],
    labels=df_q3["payment_method"],
    autopct="%1.1f%%",
    colors=colors,
    startangle=90,
    wedgeprops={"edgecolor": "white", "linewidth": 2}
)
axes[0].set_title("Participación por Método de Pago\n(% de transacciones)", fontweight="bold")

# Barras — ventas totales
axes[1].bar(df_q3["payment_method"], df_q3["total_ventas"], color=colors, width=0.5)
axes[1].set_title("Ventas Totales por Método de Pago", fontweight="bold")
axes[1].set_ylabel("Ventas Totales (USD)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
for i, (v, label) in enumerate(zip(df_q3["total_ventas"], df_q3["payment_method"])):
    axes[1].text(i, v * 1.01, f"${v:,.0f}", ha="center", fontsize=9)

plt.suptitle("Análisis de Métodos de Pago", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("fig3_metodos_pago.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Figura 3 guardada")


## 4.4 Ventas por centro comercial — Semana vs Fin de semana

In [ ]:
# ─── GRÁFICA 4: VENTAS POR MALL Y TIPO DE DÍA ───────────────────────────────

df_pivot = df_q5.pivot(index="shopping_mall", columns="tipo_dia", values="total_ventas").fillna(0)
df_pivot = df_pivot.sort_values("Entre semana", ascending=False)

x = range(len(df_pivot))
width = 0.38
fig, ax = plt.subplots(figsize=(14, 6))

bars1 = ax.bar([i - width/2 for i in x], df_pivot.get("Entre semana", 0),
               width=width, label="Entre semana", color="#42A5F5")
bars2 = ax.bar([i + width/2 for i in x], df_pivot.get("Fin de semana", 0),
               width=width, label="Fin de semana", color="#FFA726")

ax.set_xticks(list(x))
ax.set_xticklabels(df_pivot.index, rotation=30, ha="right", fontsize=9)
ax.set_title("Ventas por Centro Comercial — Entre Semana vs Fin de Semana", fontweight="bold")
ax.set_ylabel("Ventas Totales (USD)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.legend()
plt.tight_layout()
plt.savefig("fig4_mall_semana.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Figura 4 guardada")


## 4.5 Distribución demográfica de clientes

In [ ]:
# ─── GRÁFICA 5: DEMOGRAFÍA DE CLIENTES ──────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Distribución de género
gender_counts = df["gender"].value_counts()
axes[0].pie(gender_counts, labels=gender_counts.index, autopct="%1.1f%%",
            colors=["#EF9A9A", "#90CAF9"], startangle=90,
            wedgeprops={"edgecolor": "white", "linewidth": 2})
axes[0].set_title("Distribución por Género", fontweight="bold")

# Histograma de edad
axes[1].hist(df["age"], bins=20, color="#7E57C2", edgecolor="white", linewidth=0.8)
axes[1].set_title("Distribución de Edad", fontweight="bold")
axes[1].set_xlabel("Edad")
axes[1].set_ylabel("Frecuencia")

# Gasto promedio por grupo de edad
df["age_group"] = pd.cut(df["age"], bins=[0,25,35,45,55,70],
                          labels=["18-25","26-35","36-45","46-55","56-70"])
age_gasto = df.groupby("age_group")["total_amount"].mean().reset_index()
axes[2].bar(age_gasto["age_group"].astype(str), age_gasto["total_amount"],
            color=sns.color_palette("viridis", len(age_gasto)))
axes[2].set_title("Gasto Promedio por Grupo de Edad", fontweight="bold")
axes[2].set_xlabel("Grupo de Edad")
axes[2].set_ylabel("Gasto Promedio (USD)")
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))

plt.suptitle("Perfil Demográfico de Clientes", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("fig5_demografia.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Figura 5 guardada")


## 4.6 Heatmap de ventas por mes y categoría

In [ ]:
# ─── GRÁFICA 6: HEATMAP TEMPORAL POR CATEGORÍA ───────────────────────────────

heatmap_data = (
    df.groupby(["month", "category"])["total_amount"]
    .sum()
    .unstack(fill_value=0)
)
heatmap_data.index = [f"Mes {m}" for m in heatmap_data.index]

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(heatmap_data, annot=True, fmt=".0f", cmap="YlOrRd",
            linewidths=0.5, ax=ax,
            annot_kws={"size": 8})
ax.set_title("Heatmap de Ventas por Mes y Categoría (USD)", fontweight="bold", fontsize=13)
ax.set_xlabel("Categoría")
ax.set_ylabel("Mes")
plt.tight_layout()
plt.savefig("fig6_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Figura 6 guardada")


---
## 4.7 Conclusiones e Insights de Negocio

A partir del análisis completo del dataset se extraen los siguientes hallazgos:

### 🛍️ Comportamiento de ventas por categoría
- Las categorías con mayor facturación representan los productos de mayor demanda. Una estrategia de **cross-selling** entre las categorías top podría incrementar el ticket promedio.
- Las categorías con alto ticket promedio pero bajo volumen de transacciones son candidatas a **campañas de activación** para aumentar su frecuencia de compra.

### 📅 Tendencias temporales
- Los picos mensuales de ventas permiten planificar **reposición de inventario** anticipada para los meses de alta demanda.
- Los meses de baja actividad son oportunidades para lanzar **promociones y descuentos** que reactiven la demanda.

### 💳 Métodos de pago
- La distribución de métodos de pago indica las **preferencias financieras** de los clientes. Si el efectivo tiene alta participación, es recomendable mantener esa infraestructura; si el crédito domina, conviene establecer alianzas con entidades bancarias.

### 🏬 Rendimiento por centro comercial
- La comparación entre semana y fin de semana revela los **malls con mayor dependencia del tráfico de fin de semana**, lo que debe considerarse en la distribución de personal y horarios.

### 👥 Perfil demográfico
- El análisis por edad permite segmentar el mercado y diseñar **ofertas diferenciadas por grupo etario**.
- Los grupos de edad con mayor gasto promedio son los más valiosos para estrategias de **retención y fidelización**.

### 📋 Recomendaciones generales
1. **Segmentación:** Implementar un programa de lealtad diferenciado por categoría de compra y grupo etario.
2. **Estacionalidad:** Planificar campañas de marketing alineadas con los picos de venta identificados.
3. **Operaciones:** Reforzar personal y logística en los centros comerciales con mayor tráfico de fin de semana.
4. **Producto:** Explorar oportunidades en categorías con alto ticket pero bajo volumen de transacciones.
